# Niên luận: Sets2Sets + Dunnhumby Sample 50K

Notebook này dựng một pipeline thực nghiệm trên **Google Colab**, không cần web UI, API hay database.

**Mục tiêu:** tái hiện mô hình **Sets2Sets** từ repository của tác giả, so sánh với baseline **Popularity**, rồi kiểm tra độ nhạy theo độ dài lịch sử đầu vào `H = 3, 5, 7` giỏ hàng.

- Dataset: Dunnhumby *Let's Get Sort-of-Real — Sample 50K customers*.
- Model chính: code Sets2Sets gốc (Hu & He, KDD 2019).
- Baseline: xếp hạng sản phẩm theo tần suất xuất hiện trong tập huấn luyện.
- Metrics: Recall@K, NDCG@K và PHR@K với `K = 20, 40` để khớp cách đánh giá trong source gốc.
- Thiết kế: sử dụng **cùng một nhóm khách hàng và cùng hai giỏ hàng mục tiêu cuối cùng** cho H=3/5/7; chỉ thay đổi số giỏ hàng lịch sử đưa vào model.

> Lưu ý: đây là code khởi đầu có kiểm soát. Source Sets2Sets được viết cho Python 3.6 / PyTorch 1.0.1 và cập nhật lần cuối năm 2019, nên notebook có một bản vá tương thích cho `torch.load`. Nếu Colab báo lỗi khác do API PyTorch mới, cần kiểm tra traceback thay vì mặc định rằng chạy được nguyên trạng.

Nguồn:
- [Repository Sets2Sets chính thức](https://github.com/HaojiHu/Sets2Sets)
- [Trang dữ liệu Dunnhumby](https://www.dunnhumby.com/source-files/)
- [A Systematical Evaluation for Next-Basket Recommendation Algorithms](https://arxiv.org/abs/2209.02892)

## 0. Cấu hình chạy

Trước tiên, chạy thử với khoảng **1.000 khách hàng và 2 epoch** để kiểm tra toàn bộ pipeline. Sau khi chạy thành công, tăng `MAX_USERS` lên `None` và `NUM_EPOCHS` lên 10–20 để thực nghiệm nghiêm túc hơn.

Bạn cần tải và giải nén Sample 50K từ Dunnhumby vào Google Drive. Thư mục cần chứa các file giao dịch CSV (thường có tên `transactions_*.csv`) và có các cột `shop_date`, `prod_code`, `cust_code`. `time.csv` không cần cho thí nghiệm này.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, csv, re, sys, json, math, random, subprocess, shutil
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# SỬA đường dẫn này cho khớp thư mục đã giải nén trên Google Drive.
DATA_DIR = '/content/drive/MyDrive/dunnhumby_50k'

# Thí nghiệm
HISTORY_LENGTHS = [3, 5, 7]
N_FUTURE = 2                  # dự đoán 2 giỏ hàng tiếp theo
K_VALUES = [20, 40]           # Sets2Sets gốc đánh giá K=20 và K=40
MAX_USERS = 1000              # chạy thử; đổi thành None để dùng toàn bộ user đủ điều kiện
NUM_EPOCHS = 2                # smoke test; dùng 10–20 cho kết quả cuối
SEED = 42
CHUNK_SIZE = 300_000

assert os.path.isdir(DATA_DIR), f'Không tìm thấy DATA_DIR: {DATA_DIR}'
print('DATA_DIR:', DATA_DIR)
print('History lengths:', HISTORY_LENGTHS)

## 1. Kiểm tra file dữ liệu và schema

Notebook chỉ đọc những CSV có đủ các cột giao dịch yêu cầu. Cách này giúp loại `time.csv` và các file phụ không liên quan mà không cần giả định mọi CSV trong thư mục đều có cùng schema.

In [ ]:
required_columns = {'shop_date', 'prod_code', 'cust_code'}
all_csv = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
if not all_csv:
    raise FileNotFoundError(f'Không thấy CSV nào trong {DATA_DIR}. Kiểm tra xem ZIP đã được giải nén chưa.')

transaction_files = []
file_schemas = []
for f in all_csv:
    try:
        header = pd.read_csv(f, nrows=0, encoding='utf-8-sig')
    except UnicodeDecodeError:
        header = pd.read_csv(f, nrows=0, encoding='latin1')
    normalized = {str(c).strip().lower() for c in header.columns}
    if required_columns.issubset(normalized):
        transaction_files.append(f)
        file_schemas.append((os.path.basename(f), list(header.columns)))

print(f'Tổng CSV tìm thấy: {len(all_csv)}')
print(f'CSV giao dịch khớp schema: {len(transaction_files)}')
if not transaction_files:
    raise ValueError(
        'Không tìm thấy file có đủ shop_date, prod_code, cust_code. '
        'Hãy kiểm tra bạn tải đúng Dunnhumby Let’s Get Sort-of-Real Sample 50K và đúng DATA_DIR.'
    )

for name, columns in file_schemas[:3]:
    print('\n', name, '\n ', columns)

# Xem ví dụ một file giao dịch
sample = pd.read_csv(transaction_files[0], nrows=5, encoding='utf-8-sig')
display(sample)

## 2. Đọc và gom thành các giỏ hàng theo khách hàng–ngày

Dữ liệu Let’s Get Sort-of-Real có các dòng sản phẩm, nhưng không có basket ID theo cấu trúc thường thấy trong các dataset giao dịch khác. Để tương thích với cách tiền xử lý của tác giả Sets2Sets, notebook xem các sản phẩm của cùng `cust_code` trong cùng `shop_date` là một **basket/set**.

Code đọc theo chunks để không phải nạp toàn bộ CSV thành một DataFrame lớn cùng lúc. Mỗi basket được lưu như một tập sản phẩm, tránh tính một sản phẩm nhiều lần trong cùng giỏ.

In [ ]:
# user -> date -> set(products)
baskets_by_user = defaultdict(lambda: defaultdict(set))
rows_read = 0

for file_idx, file_path in enumerate(transaction_files, start=1):
    # Dùng tên cột nguyên bản để usecols hoạt động dù chữ hoa/thường khác nhau.
    header = pd.read_csv(file_path, nrows=0, encoding='utf-8-sig')
    original_by_norm = {str(c).strip().lower(): c for c in header.columns}
    usecols = [original_by_norm['shop_date'], original_by_norm['prod_code'], original_by_norm['cust_code']]

    try:
        reader = pd.read_csv(
            file_path, usecols=usecols, dtype=str, chunksize=CHUNK_SIZE,
            encoding='utf-8-sig', keep_default_na=False
        )
        for chunk in reader:
            chunk.columns = [str(c).strip().lower() for c in chunk.columns]
            chunk = chunk.rename(columns={
                'shop_date': 'shop_date', 'prod_code': 'prod_code', 'cust_code': 'cust_code'
            })
            for col in ['shop_date', 'prod_code', 'cust_code']:
                chunk[col] = chunk[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
            chunk = chunk[(chunk['shop_date'] != '') & (chunk['prod_code'] != '') & (chunk['cust_code'] != '')]
            rows_read += len(chunk)

            # Gom các sản phẩm theo khách hàng và ngày trong từng chunk rồi hợp nhất.
            grouped = chunk.groupby(['cust_code', 'shop_date'], sort=False)['prod_code'].agg(set)
            for (uid, date), products in grouped.items():
                baskets_by_user[str(uid)][str(date)].update(str(p) for p in products if str(p).strip())
    except UnicodeDecodeError:
        # Fallback hiếm gặp nếu file không phải UTF-8.
        reader = pd.read_csv(
            file_path, usecols=usecols, dtype=str, chunksize=CHUNK_SIZE,
            encoding='latin1', keep_default_na=False
        )
        for chunk in reader:
            chunk.columns = [str(c).strip().lower() for c in chunk.columns]
            for col in ['shop_date', 'prod_code', 'cust_code']:
                chunk[col] = chunk[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
            chunk = chunk[(chunk['shop_date'] != '') & (chunk['prod_code'] != '') & (chunk['cust_code'] != '')]
            rows_read += len(chunk)
            grouped = chunk.groupby(['cust_code', 'shop_date'], sort=False)['prod_code'].agg(set)
            for (uid, date), products in grouped.items():
                baskets_by_user[str(uid)][str(date)].update(str(p) for p in products if str(p).strip())

print('Số dòng đã đọc:', f'{rows_read:,}')
print('Số khách hàng:', f'{len(baskets_by_user):,}')
print('Số cặp khách hàng–ngày:', f'{sum(len(v) for v in baskets_by_user.values()):,}')
print('Ví dụ customer:', next(iter(baskets_by_user.keys())))

## 3. Tạo input của Sets2Sets cho H=3, 5, 7

Để đánh giá ảnh hưởng của lịch sử đầu vào mà không đổi target, ta giữ lại những khách hàng có ít nhất `max(HISTORY_LENGTHS) + N_FUTURE` ngày mua. Với từng khách hàng:

- **Target:** hai basket cuối cùng trong chuỗi thời gian.
- **History:** lấy H basket ngay trước target, với H=3, 5 hoặc 7.
- **Cohort:** cùng danh sách khách hàng cho cả ba thí nghiệm.
- **Split:** source Sets2Sets chia user thành khoảng 72% train, 8% validation, 20% test; baseline sẽ dùng cùng thứ tự user để có split tương ứng.

Đây là biến thể tiền xử lý để thực hiện thí nghiệm độ dài lịch sử; model vẫn lấy nguyên kiến trúc từ repository gốc. Nó không giống hệt script tiền xử lý gốc vốn cố định 5 basket history và 3 basket future.

In [ ]:
max_h = max(HISTORY_LENGTHS)
min_baskets_needed = max_h + N_FUTURE
eligible_users = [
    uid for uid, date_map in baskets_by_user.items()
    if len(date_map) >= min_baskets_needed
]
eligible_users = sorted(eligible_users)
print(f'Khách hàng đủ ít nhất {min_baskets_needed} ngày mua: {len(eligible_users):,}')
if len(eligible_users) < 100:
    raise ValueError('Quá ít khách hàng đủ điều kiện. Kiểm tra schema/dữ liệu đã nạp.')

# Xáo trộn user có seed để không chia train/test theo thứ tự mã khách hàng.
rng = random.Random(SEED)
rng.shuffle(eligible_users)
if MAX_USERS is not None:
    users_order = eligible_users[:min(int(MAX_USERS), len(eligible_users))]
else:
    users_order = eligible_users

EXP_DIR = '/content/experiment_data'
os.makedirs(EXP_DIR, exist_ok=True)

history_by_h = {}
future_by_h = {}
files_by_h = {}
summary_rows = []

for h in HISTORY_LENGTHS:
    history_by_h[h] = {}
    future_by_h[h] = {}
    hist_path = os.path.join(EXP_DIR, f'dunnhumby_H{h}_history.csv')
    fut_path = os.path.join(EXP_DIR, f'dunnhumby_H{h}_future.csv')

    with open(hist_path, 'w', newline='', encoding='utf-8') as f_hist, \
         open(fut_path, 'w', newline='', encoding='utf-8') as f_fut:
        wh = csv.writer(f_hist)
        wf = csv.writer(f_fut)
        header = ['CUSTOMER_ID', 'ORDER_NUMBER', 'MATERIAL_NUMBER']
        wh.writerow(header)
        wf.writerow(header)

        for uid in users_order:
            date_map = baskets_by_user[uid]
            dates = sorted(date_map.keys())  # shop_date định dạng YYYYMMDD; sắp xếp chuỗi cũng đúng thứ tự thời gian
            target_dates = dates[-N_FUTURE:]
            hist_dates = dates[-(h + N_FUTURE):-N_FUTURE]
            hist_seq = [set(date_map[d]) for d in hist_dates]
            fut_seq = [set(date_map[d]) for d in target_dates]
            history_by_h[h][uid] = hist_seq
            future_by_h[h][uid] = fut_seq

            for date, products in zip(hist_dates, hist_seq):
                for pid in sorted(products):
                    wh.writerow([uid, date, pid])
            for date, products in zip(target_dates, fut_seq):
                for pid in sorted(products):
                    wf.writerow([uid, date, pid])

    files_by_h[h] = (hist_path, fut_path)
    summary_rows.append({
        'History length H': h,
        'Users': len(users_order),
        'History baskets/user': h,
        'Target baskets/user': N_FUTURE,
        'Unique products in history+target': len({p for uid in users_order for b in history_by_h[h][uid] + future_by_h[h][uid] for p in b}),
        'History CSV': hist_path,
        'Future CSV': fut_path,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## 4. Baseline Popularity

Popularity xếp hạng sản phẩm theo số basket xuất hiện trong phần train. Để gần với split của Sets2Sets, code sử dụng cùng thứ tự user:

- 72% đầu: train; dùng lịch sử và target của các user train để tính tần suất.
- 8% tiếp theo: validation (không dùng để tính baseline).
- 20% cuối: test.

Với mỗi target basket, tính Recall@K và NDCG@K. `PHR@K` được tính theo kiểu person-wise hit: một user được tính hit khi **cả hai** target basket đều có ít nhất một sản phẩm trong Top-K, giống điều kiện hit của source Sets2Sets.

In [ ]:
def ranking_metrics_at_k(ranked_items, ground_truth, k):
    gt = set(ground_truth)
    if not gt:
        return 0.0, 0.0, 0
    topk = ranked_items[:k]
    hits = [1 if item in gt else 0 for item in topk]
    recall = sum(hits) / len(gt)
    dcg = sum(rel / math.log2(rank + 2) for rank, rel in enumerate(hits))
    ideal_len = min(len(gt), k)
    idcg = sum(1 / math.log2(rank + 2) for rank in range(ideal_len))
    ndcg = dcg / idcg if idcg else 0.0
    hit = int(any(hits))
    return recall, ndcg, hit


def popularity_eval(h):
    n = len(users_order)
    train_end = int((4 / 5) * n * 0.9)  # source: 72% train
    test_start = int((4 / 5) * n)      # source: 20% test bắt đầu ở 80%
    train_users = users_order[:train_end]
    test_users = users_order[test_start:]

    freq = Counter()
    for uid in train_users:
        for basket in history_by_h[h][uid] + future_by_h[h][uid]:
            freq.update(basket)  # basket là set nên mỗi item tính tối đa một lần/basket
    ranked_items = [pid for pid, _ in sorted(freq.items(), key=lambda pair: (-pair[1], pair[0]))]

    rows = []
    for k in K_VALUES:
        recalls, ndcgs = [], []
        person_hits = []
        for uid in test_users:
            user_basket_hits = []
            for target_basket in future_by_h[h][uid]:
                recall, ndcg, hit = ranking_metrics_at_k(ranked_items, target_basket, k)
                recalls.append(recall)
                ndcgs.append(ndcg)
                user_basket_hits.append(hit)
            person_hits.append(int(len(user_basket_hits) == N_FUTURE and all(user_basket_hits)))
        rows.append({
            'History length H': h,
            'Model': 'Popularity',
            'K': k,
            'Recall': float(np.mean(recalls)) if recalls else 0.0,
            'NDCG': float(np.mean(ndcgs)) if ndcgs else 0.0,
            'PHR': float(np.mean(person_hits)) if person_hits else 0.0,
            'Test users': len(test_users),
            'Model selection': 'N/A — non-parametric baseline',
        })
    return rows

baseline_rows = []
for h in HISTORY_LENGTHS:
    baseline_rows.extend(popularity_eval(h))

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)

## 5. Tải source Sets2Sets và vá tương thích `torch.load`

Source gốc hiện gọi `torch.load()` để nạp nguyên object model. Ở PyTorch mới, `weights_only` mặc định có thể gây lỗi khi nạp các checkpoint kiểu này. Notebook tạo một bản chạy riêng `Sets2Sets_colab.py` và truyền `weights_only=False` **chỉ với checkpoint do chính source Sets2Sets vừa huấn luyện tạo ra**.

Không sửa kiến trúc model. Thay đổi `topk = 400` thành `min(400, output_size)` chỉ giúp tránh lỗi nếu vocabulary nhỏ hơn 400 item. Notebook cũng bổ sung sentinel kết thúc cho user cuối cùng trong file parser; source gốc chỉ thêm sentinel này khi chuyển sang user kế tiếp, làm user cuối bị loại khỏi test split.

In [ ]:
REPO_DIR = '/content/Sets2Sets'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/HaojiHu/Sets2Sets.git', REPO_DIR], check=True)

%pip -q install bottleneck

original_py = os.path.join(REPO_DIR, 'Sets2Sets.py')
patched_py = os.path.join(REPO_DIR, 'Sets2Sets_colab.py')
with open(original_py, 'r', encoding='utf-8') as f:
    src = f.read()

if 'num_iter = 20' not in src:
    raise RuntimeError('Không tìm thấy dòng num_iter = 20 trong source; cần kiểm tra source hiện tại.')
src = src.replace('num_iter = 20', f'num_iter = {NUM_EPOCHS}', 1)
src = src.replace('torch.load(encoder_pathes)', 'torch.load(encoder_pathes, weights_only=False)')
src = src.replace('torch.load(decoder_pathes)', 'torch.load(decoder_pathes, weights_only=False)')
src = src.replace('topk = 400', 'topk = min(400, output_size)')

# Đóng sequence cuối cùng bằng sentinel [-1], giống các sequence trước.
old_eof = "if last_pid_date not in '*' and last_pid not in '-1':\n                data_chunk[file_id][last_pid].append(np.sort(feature_vector))"
new_eof = old_eof + "\n                data_chunk[file_id][last_pid].append([-1])"
if old_eof in src:
    src = src.replace(old_eof, new_eof, 1)
else:
    print('WARNING: không tìm được đoạn EOF để thêm end sentinel; kiểm tra source Sets2Sets hiện tại.')

with open(patched_py, 'w', encoding='utf-8') as f:
    f.write(src)

import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
print('Patched script:', patched_py)
print('Epochs in this run:', NUM_EPOCHS)

## 6. Huấn luyện và đánh giá Sets2Sets

Source gốc đánh giá `K=20` và `K=40`. Với mỗi H, notebook huấn luyện một model riêng; khi đánh giá, lấy kết quả test tại epoch được chọn theo **validation NDCG** (tránh chọn epoch bằng test set).

> Thử nghiệm 1.000 user / 2 epoch chỉ xác nhận pipeline chạy. Để báo cáo kết quả cuối, tăng số user và số epoch, lưu log, và chạy lại từ đầu với seed cố định.

In [ ]:
import re


def parse_sets2sets_log(log_text, h):
    parsed = []
    # Mỗi block bắt đầu tại "k = 20" hoặc "k = 40".
    sections = list(re.finditer(r'(?m)^k\s*=\s*(\d+)\s*$', log_text))
    for idx, match in enumerate(sections):
        k = int(match.group(1))
        start = match.end()
        end = sections[idx + 1].start() if idx + 1 < len(sections) else len(log_text)
        block = log_text[start:end]
        # Chọn epoch theo validation NDCG, nhưng các metric được in ra là trên test.
        metric_match = re.search(
            r'max valid ndcg results:\s*Epoch:\s*(\d+)\s*'
            r'recall:\s*([0-9.eE+-]+)\s*ndcg:\s*([0-9.eE+-]+)\s*phr:\s*([0-9.eE+-]+)',
            block, flags=re.S | re.I
        )
        if metric_match:
            epoch, recall, ndcg, phr = metric_match.groups()
            parsed.append({
                'History length H': h,
                'Model': 'Sets2Sets',
                'K': k,
                'Recall': float(recall),
                'NDCG': float(ndcg),
                'PHR': float(phr),
                'Test users': int(len(users_order) - int((4 / 5) * len(users_order))),
                'Model selection': f'epoch {epoch}, selected by validation NDCG',
            })
    return parsed

sets2sets_rows = []
model_logs = {}

for h in HISTORY_LENGTHS:
    hist_path, fut_path = files_by_h[h]
    # Source nối './' + đường dẫn nên truyền đường dẫn tương đối tính từ REPO_DIR.
    hist_rel = os.path.relpath(hist_path, REPO_DIR)
    fut_rel = os.path.relpath(fut_path, REPO_DIR)
    model_id = f'Dunnhumby_H{h}'

    print('\n' + '=' * 72)
    print(f'TRAIN Sets2Sets | H={h} | users={len(users_order)} | epochs={NUM_EPOCHS}')
    train_cmd = [sys.executable, 'Sets2Sets_colab.py', hist_rel, fut_rel, model_id, str(N_FUTURE), '1']
    subprocess.run(train_cmd, cwd=REPO_DIR, check=True)

    print(f'\nTEST Sets2Sets | H={h}')
    test_cmd = [sys.executable, 'Sets2Sets_colab.py', hist_rel, fut_rel, model_id, str(N_FUTURE), '0']
    completed = subprocess.run(test_cmd, cwd=REPO_DIR, text=True, capture_output=True)
    print(completed.stdout[-12000:])
    if completed.returncode != 0:
        print(completed.stderr[-12000:])
        raise subprocess.CalledProcessError(completed.returncode, test_cmd, output=completed.stdout, stderr=completed.stderr)

    model_logs[h] = completed.stdout
    parsed = parse_sets2sets_log(completed.stdout, h)
    if not parsed:
        with open(os.path.join(REPO_DIR, f'test_log_H{h}.txt'), 'w', encoding='utf-8') as f:
            f.write(completed.stdout)
        raise RuntimeError(
            f'Không parse được metric cho H={h}. Log đã lưu ở {REPO_DIR}/test_log_H{h}.txt; '
            'kiểm tra định dạng output của source Sets2Sets.'
        )
    sets2sets_rows.extend(parsed)

sets2sets_df = pd.DataFrame(sets2sets_rows)
display(sets2sets_df)

## 7. Tổng hợp kết quả và phân tích ảnh hưởng của độ dài lịch sử

Đọc bảng theo từng `H` và `K`. Nếu Sets2Sets vượt Popularity ở H=7 nhưng không vượt ở H=3, có thể nói model nhạy với lượng lịch sử sẵn có **trong thiết lập và dataset này**. Không nên khẳng định nó tốt hơn trên mọi dataset.

In [ ]:
if sets2sets_df.empty:
    raise ValueError('Chưa có kết quả Sets2Sets để tổng hợp.')

results_df = pd.concat([baseline_df, sets2sets_df], ignore_index=True)
results_df = results_df.sort_values(['K', 'History length H', 'Model']).reset_index(drop=True)
display(results_df)

pivot_recall = results_df.pivot_table(index=['History length H', 'K'], columns='Model', values='Recall').reset_index()
pivot_ndcg = results_df.pivot_table(index=['History length H', 'K'], columns='Model', values='NDCG').reset_index()

if {'Popularity', 'Sets2Sets'}.issubset(pivot_recall.columns):
    pivot_recall['Sets2Sets - Popularity (Recall)'] = pivot_recall['Sets2Sets'] - pivot_recall['Popularity']
if {'Popularity', 'Sets2Sets'}.issubset(pivot_ndcg.columns):
    pivot_ndcg['Sets2Sets - Popularity (NDCG)'] = pivot_ndcg['Sets2Sets'] - pivot_ndcg['Popularity']

print('Chênh lệch Recall: dương nghĩa Sets2Sets tốt hơn baseline theo Recall ở thiết lập đó.')
display(pivot_recall)
print('Chênh lệch NDCG: dương nghĩa Sets2Sets xếp hạng tốt hơn baseline theo NDCG ở thiết lập đó.')
display(pivot_ndcg)

for metric in ['Recall', 'NDCG', 'PHR']:
    for k in K_VALUES:
        subset = results_df[results_df['K'] == k]
        plt.figure(figsize=(7, 4))
        for model_name in subset['Model'].unique():
            sub = subset[subset['Model'] == model_name].sort_values('History length H')
            plt.plot(sub['History length H'], sub[metric], marker='o', label=model_name)
        plt.xlabel('Số basket lịch sử đầu vào (H)')
        plt.ylabel(f'{metric}@{k}' if metric != 'PHR' else f'PHR@{k}')
        plt.title(f'{metric} theo độ dài lịch sử (K={k})')
        plt.xticks(HISTORY_LENGTHS)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

OUT_CSV = '/content/sets2sets_dunnhumby_results.csv'
results_df.to_csv(OUT_CSV, index=False)
print('Đã lưu:', OUT_CSV)

## 8. Kiểm tra trước khi dùng kết quả trong báo cáo

1. **Smoke test không phải kết quả cuối:** mặc định 1.000 user và 2 epoch chỉ nhằm xác nhận pipeline; cần chạy nhiều user/epoch hơn cho bảng kết quả chính thức.
2. **Ghi rõ cách chia dữ liệu:** user được xáo trộn với seed cố định; khoảng 72% train, 8% validation, 20% test, theo cách chia user trong source gốc.
3. **Giữ nguyên target khi so H:** target là hai basket cuối cùng của cùng khách hàng; model chỉ nhận 3, 5 hoặc 7 basket ngay trước target.
4. **Popularity là baseline không cá nhân hóa:** tần suất item tính trên lịch sử và các target đã biết của user train, không dùng tương tác của user test.
5. **PHR:** trong notebook này, một user được tính hit nếu cả hai target basket đều có ít nhất một item trong Top-K; cần mô tả định nghĩa này trong báo cáo.
6. **Nêu rõ giới hạn:** dữ liệu Dunnhumby là dữ liệu giả lập theo mẫu hành vi bán lẻ, và code Sets2Sets gốc khá cũ. Lưu lại phiên bản Python/PyTorch, seed, số user, epoch và lỗi/sửa đổi khi thực nghiệm.
7. **Kết quả không được bảo đảm trước:** nếu Sets2Sets không vượt Popularity, đó vẫn là kết quả hợp lệ và cần được phân tích trung thực.

Tài liệu nên trích dẫn:
- Hu, H. & He, X. (2019). *Sets2Sets: Learning from Sequential Sets with Neural Networks*. KDD 2019.
- Shao et al. (2022). *A Systematical Evaluation for Next-Basket Recommendation Algorithms*.